# zedstat example notebook

This notebook is a full, commented walkthrough of the current `zedstat.py` workflow.

It covers:

- loading data and checking the active `zedstat.py` interface
- computing the empirical ROC table
- smoothing with or without convexification
- computing threshold-based measures and analytic bounds
- using the updated NaN-safe final-table API, including `zt.get(0)`, `zt.get(bounds=True, interpolate=True)`, and `zt.get_full()`
- using the updated `getBounds(...)` options for direct PPV bounds, expected-count masking, PPV-bound smoothing, and dominated-LR masking
- using `usample(..., recompute_measures=True)` so derived ratios are recomputed from FPR/TPR rather than interpolated directly
- plotting ROC, PRC, and LR-space curves with low-opacity confidence envelopes
- extracting nondominated likelihood-ratio frontiers
- extracting operating points
- mapping score to threshold PPV
- running held-out isotonic calibration and querying calibrated patient-level probabilities
- estimating AUC intervals and sample-size requirements

The main intended output table is `perf_df`. In the updated package, the preferred construction is:

```python
perf_df = zt.get(0)
```

This is shorthand for a final display table with nominal measures joined to upper and lower bounds, followed by numeric interpolation and bound re-enforcement. It is not a literal zero fill. Endpoint likelihood-ratio values that are undefined should remain undefined unless you explicitly choose a `fillna` value.


## 1. User parameters

Edit only this cell first. Everything else should run from these settings.

In [ ]:
from pathlib import Path
import numpy as np
# ------------------------------------------------------------------
# FILE / COLUMN SETTINGS
# ------------------------------------------------------------------
data_path = Path("./PREDICTIONS_NATIONAL_SAMPLE.parquet")   # change to your file
data_path = Path("./SUICIDALITY-M25504W6WS.parquet")   # change to your file
read_method = "parquet"                 # one of: 'parquet', 'csv'
score_col = "predicted_risk"
label_col = "target"

# ------------------------------------------------------------------
# PROBLEM SETTINGS
# ------------------------------------------------------------------
deployment_prevalence = 0.10            # prevalence to use for PPV / NPV style measures
lower_score_is_risk = False             # set True only if smaller score = higher risk

# ------------------------------------------------------------------
# ROC / MEASURE SETTINGS
# ------------------------------------------------------------------
use_convexify = False                   # True => use ROC upper hull before interpolation
smooth_step = 0.001                     # FPR grid spacing for smooth()
precision = 3                           # decimal precision for usample()
alpha = 0.05                            # confidence level parameter
lr_fpr_floor = 0.001                    # censor LR+ / LR- when FPR is below this

# ------------------------------------------------------------------
# CALIBRATION SETTINGS
# ------------------------------------------------------------------
calibration_test_size = 0.25            # held-out test fraction
calibration_n_bins = 100                # number of bins in calibration plot/table
calibration_n_boot = 1000               # bootstrap replicates for held-out calibration metrics
calibration_target_prevalence = None    # None => use observed prevalence; set a float for reweighted deployment prevalence
random_state = 4

# Calibration output files written by zedstat.calibration
calibration_df_path = "calibration_df.csv"
calibration_plot_path = "calibration.pdf"

# ------------------------------------------------------------------
# THRESHOLD / INTERPRETATION SETTINGS
# ------------------------------------------------------------------
example_scores = [0.02, 0.05, 0.10]     # example score values to query
interpret_fpr = 0.01                    # operating point to interpret
interpret_positive_cases = 100
five_yr_survival = None                 # set numeric value if you want NNS

# ------------------------------------------------------------------
# SAMPLE SIZE SETTINGS
# ------------------------------------------------------------------
delta_auc_grid = np.linspace(0.01, 0.10, 10)
target_auc_for_planning = None          # None => use nominal AUC estimated from current data
sample_size_target_auc = None           # None => use current nominal AUC


## 2. Imports and module sanity check

This cell prints the active file path and the signatures of the main entry points.  
That is useful because several earlier copies of `zedstat.py` in this thread had slightly different interfaces.

In [ ]:
import inspect
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from zedstat import zedstat
from zedstat import calibration
# importlib.reload(zedstat)
# importlib.reload(calibration)

print("Active zedstat.py:", zedstat.__file__)
print("Active calibration.py:", calibration.__file__)
print()

print("processRoc.__init__ signature:")
print(inspect.signature(zedstat.processRoc.__init__))
print()

print("processRoc.get signature:")
print(inspect.signature(zedstat.processRoc.get))
print()

print("processRoc.usample signature:")
print(inspect.signature(zedstat.processRoc.usample))
print()

print("processRoc.getBounds signature:")
print(inspect.signature(zedstat.processRoc.getBounds))
print()

print("pipeline signature:")
print(inspect.signature(zedstat.pipeline))
print()

print("Has get_full:", hasattr(zedstat.processRoc, "get_full"))
print("Has object-level lr_frontier:", hasattr(zedstat.processRoc, "lr_frontier"))
print("Has module-level lr_nondominated_frontier:", hasattr(zedstat, "lr_nondominated_frontier"))


## 3. Load the data

This notebook supports parquet and CSV for convenience.

In [ ]:
if read_method == "parquet":
    df = pd.read_parquet(data_path)
elif read_method == "csv":
    df = pd.read_csv(data_path)
else:
    raise ValueError("read_method must be 'parquet' or 'csv'")

print(df.shape)
display(df.head())

if score_col not in df.columns:
    raise KeyError(f"Missing score column: {score_col}")

if label_col not in df.columns:
    raise KeyError(f"Missing label column: {label_col}")

print("Columns OK.")

## 4. Compute the empirical ROC table

`genroc(...)` creates the empirical threshold/FPR/TPR table directly from raw scores and labels.

In [ ]:
rf, total_samples, positive_samples = zedstat.genroc(
    df,
    risk=score_col,
    target=label_col,
)

print("total_samples =", total_samples)
print("positive_samples =", positive_samples)
display(rf.head())

## 5. Build the main `processRoc` object

This block is written to handle both cases:
- newer `zedstat.py` versions where `lr_fpr_floor` is accepted directly in the constructor
- older nearby variants where it is set later with `set_lr_fpr_floor(...)`

In [ ]:
proc_sig = inspect.signature(zedstat.processRoc.__init__)

proc_kwargs = dict(
    df=rf,
    prevalence=deployment_prevalence,
    total_samples=total_samples,
    positive_samples=positive_samples,
    alpha=alpha,
)

if "lr_fpr_floor" in proc_sig.parameters:
    proc_kwargs["lr_fpr_floor"] = lr_fpr_floor

zt = zedstat.processRoc(**proc_kwargs)

if hasattr(zt, "set_lr_fpr_floor"):
    zt.set_lr_fpr_floor(lr_fpr_floor)
elif not hasattr(zt, "lr_fpr_floor"):
    # This fallback keeps the notebook usable even if a nearby version
    # stores the threshold as a plain attribute.
    zt.lr_fpr_floor = lr_fpr_floor

print("processRoc object created.")

## 6. Smooth the ROC, compute measures, and derive analytic bounds

This cell exercises the updated main workflow.

`zt.smooth(...)` places the ROC curve on an FPR grid. `convexify=False` keeps the empirical ROC shape and interpolates it; `convexify=True` first replaces the empirical curve with the upper ROC hull.

`zt.allmeasures(...)` computes threshold-level TPR, PPV, accuracy, NPV, LR+, and LR- at the deployment prevalence.

`zt.usample(..., recompute_measures=True)` is important in the updated package. It interpolates the primary ROC coordinates and threshold-like fields, then recomputes derived measures such as PPV, NPV, LR+, and LR-. This avoids creating artificial likelihood-ratio branches by directly interpolating ratios.

`zt.getBounds(...)` now supports direct PPV intervals, minimum expected-count masks, optional monotone smoothing of PPV bounds, and optional dominated-LR masking. These settings are the recommended defaults for cleaner plots and more stable output tables.

`zt.get(0)` is the compact final-output call. It joins `zt.df_lim["U"]` and `zt.df_lim["L"]` to the nominal table, interpolates display gaps, clips probability-like columns to `[0, 1]`, and re-enforces lower/nominal/upper consistency. Use `zt.get()` with no arguments only when you want the historical nominal-only table.


In [ ]:
zt.smooth(STEP=smooth_step, convexify=use_convexify)
zt.allmeasures(prevalence=deployment_prevalence, interpolate=True)

# Updated behavior: when recompute_measures=True, only the primary ROC
# coordinates are interpolated and the derived measures are recomputed.
zt.usample(precision=precision, recompute_measures=True)

# Build bounds using only arguments supported by the active zedstat.py.
# This keeps the notebook usable across nearby development versions while
# documenting the current recommended options.
bounds_sig = inspect.signature(zt.getBounds)
bounds_kwargs = dict(
    ppv_ci_method="direct",
    min_expected_flags=10.0,
    min_expected_fp=5.0,
    min_expected_tn=5.0,
    enforce_bounds=True,
    smooth_ppv_bounds=True,
    mask_dominated_lr=True,
    lr_min_plus=1.001,
)
bounds_kwargs = {k: v for k, v in bounds_kwargs.items() if k in bounds_sig.parameters}
zt.getBounds(**bounds_kwargs)

# Preferred final table call in the updated package.
# zt.get(0) means: joined nominal + bounds table, NaN-safe interpolation,
# and display-bound consistency. It does not mean fill LR endpoints with zero.
perf_df = zt.get(0)
display(perf_df.head())
print(perf_df.columns.tolist())


## 7. Quick look at the resulting performance table

The combined `perf_df` contains the nominal threshold-level measures and their upper/lower analytic bounds.

Useful retrieval patterns are:

```python
zt.get()                                  # nominal table only, historical behavior
zt.get(0)                                 # preferred final display table
zt.get(bounds=True, interpolate=True)     # explicit version of the final display table
zt.get_full()                             # convenience wrapper, if available
```

Use `zt.get(0)` or `zt.get_full()` for plotting and export. Use `zt.get()` when you intentionally want only the nominal table.


In [ ]:
summary_cols = [c for c in ["threshold", "tpr", "ppv", "npv", "LR+", "LR-", "tpr_upper", "ppv_upper", "tpr_lower", "ppv_lower"] if c in perf_df.columns]
display(perf_df[summary_cols].head(10))

## 7b. Demonstrate the updated final-table accessors

This cell explicitly demonstrates the new access patterns. It is safe to leave in the notebook because it only reads existing `zt` state.


In [ ]:
# Nominal-only table, preserving the historical behavior.
nominal_only_df = zt.get()

# Explicit final display table call.
try:
    perf_df_explicit = zt.get(bounds=True, interpolate=True)
except TypeError:
    perf_df_explicit = perf_df.copy()

# Convenience wrapper if the active package provides it.
if hasattr(zt, "get_full"):
    perf_df_full = zt.get_full()
else:
    perf_df_full = perf_df_explicit.copy()

print("nominal_only_df shape:", nominal_only_df.shape)
print("perf_df shape:", perf_df.shape)
print("perf_df_explicit shape:", perf_df_explicit.shape)
print("perf_df_full shape:", perf_df_full.shape)

display(perf_df_full.head())


## 8. Plot ROC, PRC, and LR-space curves with filled bounds

The ROC panel uses the nominal FPR grid directly.  
The PRC and LR panels are drawn as low-opacity polygons between lower and upper curves.

In [ ]:
import matplotlib.pyplot as plt

pf = perf_df

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC
axes[0].fill_between(
    pf["fpr"],
    pf["tpr_lower"],
    pf["tpr_upper"],
    alpha=0.15,
)
axes[0].plot(pf["fpr"], pf["tpr"], lw=2)
axes[0].plot([0, 1], [0, 1], "--", color="0.5", lw=1)
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC curve")
axes[0].grid(alpha=0.3)


# Precision-recall style curve
axes[1].fill_between(
    pf["tpr"],
    pf["ppv_lower"],
    pf["ppv_upper"],
    alpha=0.15,
)
axes[1].plot(pf["tpr"], pf["ppv"], lw=2)
axes[1].set_xlabel("Recall / TPR")
axes[1].set_ylabel("Precision / PPV")
axes[1].set_title("Precision-recall style curve")
axes[1].grid(alpha=0.3)


# Likelihood-ratio curve
axes[2].fill_between(
    pf["LR-"],
    pf["LR+_lower"],
    pf["LR+_upper"],
    alpha=0.15,
)
axes[2].plot(pf["LR-"], pf["LR+"], lw=2)
axes[2].set_xlabel("LR-")
axes[2].set_ylabel("LR+")
axes[2].set_title("Likelihood-ratio curve")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Query threshold-level PPV at selected scores

This is **not** patient-level calibrated probability.  
It is the PPV associated with the threshold equal to that score.

In [ ]:
threshold_ppv = zt.score_to_threshold_ppv(
    example_scores,
    regen=False,   # use the performance table already built above
)

threshold_ppv_df = pd.DataFrame({
    "score": example_scores,
    "threshold_ppv": np.asarray(threshold_ppv, dtype=float),
})

display(threshold_ppv_df)

## 10. Run held-out isotonic calibration analysis

This path now uses `from zedstat import calibration` rather than the older
`zt.fit_score_calibration(...)` interface.

The workflow is:

1. split the data into train and test sets  
2. fit isotonic calibration on the training split only  
3. evaluate on the held-out test set  
4. report AUC, Brier score, calibration intercept, calibration slope, and bootstrap confidence intervals  
5. save the calibration table and calibration plot to the filenames specified above

The most important user-tunable parameters here are:

- `calibration_test_size`
- `calibration_n_bins`
- `calibration_n_boot`
- `calibration_df_path`
- `calibration_plot_path`


In [ ]:
res = calibration.heldout_isotonic_calibration_with_bootstrap(
    df,
    score_col=score_col,
    label_col=label_col,
    test_size=calibration_test_size,
    random_state=random_state,
    lower_score_is_risk=lower_score_is_risk,
    target_prevalence=calibration_target_prevalence,
    n_bins=calibration_n_bins,
    n_boot=calibration_n_boot,
    calibration_df_path=calibration_df_path,
    plot=calibration_plot_path,
)

cal_summary = res["summary"]
cal_df = res["calibration_table"]

print(cal_summary)
display(cal_df)


## 11. Check the saved calibration plot

The calibration helper writes the plot directly to `calibration_plot_path`.
If the file is an image, this cell will also preview it inline.


In [ ]:
from pathlib import Path

plot_path = res.get("plot_file", None)

if plot_path is None:
    print("No calibration plot file was written.")
else:
    plot_path = Path(plot_path)
    print("Calibration plot saved to:", plot_path.resolve())
    if plot_path.suffix.lower() in {".png", ".jpg", ".jpeg"}:
        from IPython.display import Image, display
        display(Image(filename=str(plot_path)))


## 12. Query calibrated patient-level probability at selected scores

The held-out calibration workflow returns the fitted isotonic model in
`res["iso_model"]`. This cell uses that mapping to convert example scores to
calibrated probabilities.


In [ ]:
score_array = np.asarray(example_scores, dtype=float)
score_for_iso = -score_array if lower_score_is_risk else score_array

calibrated_probs = res["iso_model"].predict(score_for_iso)

calibrated_df = pd.DataFrame({
    "score": example_scores,
    "calibrated_probability": np.asarray(calibrated_probs, dtype=float),
})

display(calibrated_df)


## 13. Analytic AUC and optional bootstrap AUC

The analytic AUC uses the built-in closed-form approximation.  
The bootstrap AUC uses the calibration re-sampling to estimate the AUC (better estimate)

In [ ]:
auc_nominal, auc_upper, auc_lower = zt.auc()
print("Analytic AUC")
print("nominal =", auc_nominal)
print("upper   =", auc_upper)
print("lower   =", auc_lower)

### Bootstrap AUC

In [ ]:
res["summary"].set_index('variable').loc['auc_calibrated_test']

## Likelihood-ratio frontier extraction

The updated package can hide dominated LR points in the plotted bounds through `mask_dominated_lr=True`. It can also return the nondominated LR frontier explicitly.

A point is kept on the LR frontier if no other threshold has both a smaller or equal LR- and a larger or equal LR+. This is useful when selecting clinically interpretable rule-in/rule-out thresholds from the LR-space plot.


In [ ]:
if hasattr(zt, "lr_frontier"):
    lr_frontier_df = zt.lr_frontier(min_lrplus=1.001)
elif hasattr(zedstat, "lr_nondominated_frontier"):
    lr_frontier_df = zedstat.lr_nondominated_frontier(perf_df, min_lrplus=1.001)
else:
    lr_frontier_df = pd.DataFrame()

print("LR frontier rows:", len(lr_frontier_df))
display(lr_frontier_df.head(20))


## 14. Operating-zone extraction

This identifies operating points that simultaneously satisfy:
- `LR+ > LRplus`
- `LR- < LRminus`

and then reports one high-precision and one high-sensitivity endpoint per requested `n`.

In [ ]:
zt.operating_zone(n=1, LRplus=10, LRminus=0.6)
display(zt._operating_zone)

## 15. Interpretation at a chosen false-positive rate

This block converts a chosen operating point into expected flag counts, false alarms, and missed cases for a hypothetical number of positives.

In [ ]:
rf_interpret, txt_interpret, resdf_interpret = zt.interpret(
    fpr=interpret_fpr,
    number_of_positives=interpret_positive_cases,
    five_yr_survival=five_yr_survival,
    factor=1,
)

display(rf_interpret)
display(resdf_interpret)

for line in txt_interpret:
    print(line)

## 16. Sample-size planning for AUC precision

`zt.samplesize(delta_auc=...)` estimates the required number of samples per class under the model's approximation.  
A smaller allowed AUC deviation requires more samples.

If `target_auc_for_planning` is left as `None`, the notebook uses the current nominal AUC.

In [ ]:
planning_auc = auc_nominal if target_auc_for_planning is None else float(target_auc_for_planning)

sample_size_rows = []
for d in delta_auc_grid:
    req = zt.samplesize(delta_auc=d, target_auc=planning_auc, alpha=.05, prevalence=.1)
    sample_size_rows.append({
        "delta_auc": d,
        "required_samples": req,
    })

sample_size_df = pd.DataFrame(sample_size_rows).sort_values("delta_auc", ascending=False)
display(sample_size_df)
sample_size_df.to_csv('sample_size_df.csv')

## 17. Plot required sample size versus allowed AUC deviation

The x-axis is the maximum tolerated AUC error.  
The y-axis is the required number of samples **per class** under the approximation used by `zedstat.samplesize(...)`.

In [ ]:
plt.figure(figsize=(6.5, 4.5))
plt.plot(sample_size_df["delta_auc"], sample_size_df["required_samples"], marker="o", linewidth=2)
plt.xlabel("Allowed AUC deviation")
plt.ylabel("Required samples")
plt.title(f"Sample-size planning at target AUC = {planning_auc:.4f}")
plt.grid(alpha=0.3)
plt.gca().set_yscale('log')
plt.tight_layout()
plt.show()

## 18. Optional direct `pipeline(...)` example

The `pipeline(...)` helper is convenient, but it exposes fewer intermediate objects than the manual path.  
Use the manual path above when you need fine-grained access to all features.

This block is guarded so it only passes parameters that the active `zedstat.py` version actually supports.

## Practical notes

Use `zt.get(0)` as the default table for plots and exports after `zt.getBounds(...)`. It returns the nominal table joined to upper and lower bounds and performs display-safe interpolation. The zero is only shorthand for the final-table mode; it is not a literal instruction to replace undefined likelihood-ratio endpoints with zero.

Use `zt.get()` with no arguments only when you want the nominal table without bounds.

Use `zt.usample(..., recompute_measures=True)` when making a uniform FPR grid. This recomputes PPV, NPV, LR+, and LR- from the resampled FPR/TPR curve rather than interpolating ratios directly.

Use `getBounds(ppv_ci_method="direct", min_expected_flags=10.0, min_expected_fp=5.0, min_expected_tn=5.0, enforce_bounds=True, smooth_ppv_bounds=True, mask_dominated_lr=True)` for stable display bounds. The expected-count thresholds suppress operating points where the denominators are too small to support reliable plotted intervals.

Use `zt.lr_frontier(...)` or `zedstat.lr_nondominated_frontier(perf_df, ...)` to extract the nondominated LR frontier.

Use `score_to_threshold_ppv(...)` when you want post-test yield at a threshold.

Use `from zedstat import calibration` followed by `heldout_isotonic_calibration_with_bootstrap(...)` when you want held-out patient-level calibration assessment, including AUC, Brier score, calibration intercept, calibration slope, and bootstrap confidence intervals.

Use `res["iso_model"]` when you want to map a raw score to a patient-level calibrated probability after fitting the held-out calibration model.
